# Potato / Tomato / Bell Pepper Classifier — Vegetable + Health

Trains a MobileNetV2 transfer-learning model on the PlantVillage dataset (color images), restricted to Potato, Tomato, and Bell Pepper classes.

**Steps:** get a Kaggle API token (kaggle.com -> Account -> Create New API Token, downloads `kaggle.json`), then run all cells top to bottom. Make sure Runtime > Change runtime type > GPU is selected.

In [ ]:
!pip install -q kaggle tensorflow

import os
from getpass import getpass

# Kaggle's newer token format (starts with KGAT_). Paste it when prompted below
# -- it will be hidden as you type/paste, and only lives in this session's memory.
# Get one at kaggle.com -> Settings -> API -> Create New Token.
token = getpass('Paste your Kaggle API token (KGAT_...): ')
os.environ['KAGGLE_API_TOKEN'] = token

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/access_token'), 'w') as f:
    f.write(token)
os.chmod(os.path.expanduser('~/.kaggle/access_token'), 0o600)

!kaggle datasets list -s plantvillage  # quick auth check

In [ ]:
# Download the dataset (this is a few GB, give it a few minutes)
!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p /content/data --force
!unzip -q /content/data/plantvillage-dataset.zip -d /content/data
!find /content/data -maxdepth 3 -type d | head -50

The dataset ships three parallel versions of every class: `color`, `grayscale`, `segmented`. We use **color** — disease symptoms are color cues (yellowing, blight rings, mold), so grayscale throws that signal away.

In [ ]:
import glob, shutil

# Locate the 'color' directory wherever the zip extracted it
color_dir_candidates = glob.glob('/content/data/**/color', recursive=True)
assert color_dir_candidates, 'color/ folder not found — check the extraction above'
COLOR_DIR = color_dir_candidates[0]
print('Using:', COLOR_DIR)

all_classes = sorted(os.listdir(COLOR_DIR))
print(len(all_classes), 'total classes in dataset')

# Keep only Potato, Tomato, Pepper(bell) classes
KEEP_PREFIXES = ('Potato', 'Tomato', 'Pepper')
target_classes = [c for c in all_classes if c.startswith(KEEP_PREFIXES)]
print('Keeping', len(target_classes), 'classes:')
for c in target_classes:
    print(' -', c)

In [ ]:
# Build a filtered directory containing only the classes we want
FILTERED_DIR = '/content/data_filtered'
if os.path.exists(FILTERED_DIR):
    shutil.rmtree(FILTERED_DIR)
os.makedirs(FILTERED_DIR)

for c in target_classes:
    src = os.path.join(COLOR_DIR, c)
    dst = os.path.join(FILTERED_DIR, c)
    os.symlink(src, dst)

for c in target_classes:
    n = len(os.listdir(os.path.join(FILTERED_DIR, c)))
    print(f'{c}: {n} images')

In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    FILTERED_DIR, validation_split=0.2, subset='training', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE)

val_ds = tf.keras.utils.image_dataset_from_directory(
    FILTERED_DIR, validation_split=0.2, subset='validation', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE)

class_names = train_ds.class_names
print(class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

In [ ]:
# Light augmentation to reduce overfitting
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base_model.trainable = False  # freeze for stage 1

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# Stage 1: train the new head only
history = model.fit(train_ds, validation_data=val_ds, epochs=6)

In [ ]:
# Stage 2: unfreeze the top of the base model and fine-tune at a low LR
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])

fine_tune_history = model.fit(train_ds, validation_data=val_ds, epochs=6)

In [ ]:
loss, acc = model.evaluate(val_ds)
print(f'Validation accuracy: {acc:.3f}')

In [ ]:
# Save model + class names, download both
import json

model.save('/content/veg_classifier.h5')
with open('/content/class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)

files.download('/content/veg_classifier.h5')
files.download('/content/class_names.json')